In [ ]:
from games import Game, GameState, alpha_beta_player
import ipywidgets as widgets
from IPython.display import display

In [ ]:
class Hexapawn(Game):

    def __init__(self):
        board = {
            (1,1): 'W',
            (1,2): 'W',
            (1,3): 'W',
            (3,1): 'B',
            (3,2): 'B',
            (3,3): 'B'
        }

        self.initial = GameState(
            to_move='W',
            utility=0,
            board=board,
            moves=self.get_moves(board, 'W')
        )

    def actions(self, state):
        return state.moves

    def result(self, state, move):

        board = state.board.copy()

        start, end = move

        player = state.to_move
        opponent = 'B' if player == 'W' else 'W'

        del board[start]
        board[end] = player

        # ชนะจากการไปถึงฝั่งตรงข้าม
        if player == 'W' and end[0] == 3:
            utility = 1

        elif player == 'B' and end[0] == 1:
            utility = -1

        else:
            next_moves = self.get_moves(
                board,
                opponent
            )

            # อีกฝ่ายไม่มีทางเดิน
            if len(next_moves) == 0:
                utility = 1 if player == 'W' else -1
            else:
                utility = 0

        return GameState(
            to_move=opponent,
            utility=utility,
            board=board,
            moves=self.get_moves(
                board,
                opponent
            )
        )

    def utility(self, state, player):

        if player == 'W':
            return state.utility

        return -state.utility

    def terminal_test(self, state):

        return (
            state.utility != 0
            or len(state.moves) == 0
        )

    def get_moves(self, board, player):

        moves = []

        direction = 1 if player == 'W' else -1
        opponent = 'B' if player == 'W' else 'W'

        for (row, col), piece in list(board.items()):

            if piece != player:
                continue

            next_row = row + direction

            if next_row < 1 or next_row > 3:
                continue

            # เดินตรง
            forward = (next_row, col)

            if forward not in board:
                moves.append(
                    ((row, col), forward)
                )

            # กินเฉียงซ้าย
            if col > 1:

                target = (
                    next_row,
                    col - 1
                )

                if board.get(target) == opponent:
                    moves.append(
                        ((row, col), target)
                    )

            # กินเฉียงขวา
            if col < 3:

                target = (
                    next_row,
                    col + 1
                )

                if board.get(target) == opponent:
                    moves.append(
                        ((row, col), target)
                    )

        return moves

In [ ]:
game = Hexapawn()

state = game.initial

print("Board:")
print(state.board)

print("\nTurn:", state.to_move)

print("\nAvailable moves:")
print(game.actions(state))

Board:
{(1, 1): 'W', (1, 2): 'W', (1, 3): 'W', (3, 1): 'B', (3, 2): 'B', (3, 3): 'B'}

Turn: W

Available moves:
[((1, 1), (2, 1)), ((1, 2), (2, 2)), ((1, 3), (2, 3))]


In [ ]:
ai_move = alpha_beta_player(game, state)

print("AI move:", ai_move)

AI move: ((1, 1), (2, 1))


In [ ]:
class HexapawnGUI:

    def __init__(self):

        self.game = Hexapawn()
        self.state = self.game.initial
        self.selected = None

        self.status = widgets.HTML()

        self.buttons = {}

        self.create_board()

        self.reset_button = widgets.Button(
            description="🔄 New Game",
            button_style="success"
        )

        self.reset_button.on_click(self.reset)

        self.update()

    def create_board(self):

        rows = []

        for row in range(3, 0, -1):

            row_buttons = []

            for col in range(1, 4):

                button = widgets.Button(
                    description="",
                    layout=widgets.Layout(
                        width="100px",
                        height="100px"
                    )
                )

                button.position = (row, col)

                button.on_click(self.click)

                self.buttons[(row, col)] = button

                row_buttons.append(button)

            rows.append(widgets.HBox(row_buttons))

        self.board = widgets.VBox(rows)

    def update(self):

        # เปิดปุ่มทั้งหมดก่อนทุกครั้ง
        for button in self.buttons.values():
            button.disabled = False

        # วาดกระดาน
        for position, button in self.buttons.items():

            piece = self.state.board.get(position, "")

            if piece == "W":
                button.description = "♙"
                button.button_style = "info"

            elif piece == "B":
                button.description = "♟"
                button.button_style = "danger"

            else:
                button.description = ""
                button.button_style = ""

        # เกมจบ
        if self.game.terminal_test(self.state):

            utility = self.game.utility(
                self.state,
                "W"
            )

            if utility == 1:
                self.status.value = "<h2>🎉 คุณชนะ!</h2>"
            else:
                self.status.value = "<h2>🤖 AI ชนะ!</h2>"

            for button in self.buttons.values():
                button.disabled = True

            return

        # ตาผู้เล่น
        if self.state.to_move == "W":

            self.status.value = (
                "<h3>🎮 ตาของคุณ (♙)</h3>"
                "<p>คลิก ♙ แล้วคลิกช่องปลายทาง</p>"
            )

        else:

            self.status.value = (
                "<h3>🤖 AI กำลังคิด...</h3>"
            )

    def click(self, button):

        # ไม่ใช่ตาเรา
        if self.state.to_move != "W":
            return

        position = button.position

        # -------------------------
        # ยังไม่ได้เลือกตัวหมาก
        # -------------------------
        if self.selected is None:

            if self.state.board.get(position) == "W":

                self.selected = position

                # ทำให้ตัวที่เลือกเป็นสีเหลือง
                button.button_style = "warning"

            return

        # -------------------------
        # เลือกช่องปลายทาง
        # -------------------------
        move = (
            self.selected,
            position
        )

        # เดินไม่ได้
        if move not in self.game.actions(self.state):

            self.selected = None
            self.update()
            return

        # -------------------------
        # White เดิน
        # -------------------------
        self.state = self.game.result(
            self.state,
            move
        )

        self.selected = None

        self.update()

        # เกมจบหลังเราเดิน
        if self.game.terminal_test(self.state):
            return

        # -------------------------
        # AI เดิน
        # -------------------------
        ai_move = alpha_beta_player(
            self.game,
            self.state
        )

        self.state = self.game.result(
            self.state,
            ai_move
        )

        self.update()

    def reset(self, button=None):

        self.state = self.game.initial
        self.selected = None

        self.update()

    def display(self):

        return widgets.VBox([
            widgets.HTML(
                "<h1>♙ Hexapawn vs Alpha-Beta AI ♟</h1>"
            ),
            self.status,
            self.board,
            self.reset_button
        ])

In [ ]:
game_ui = HexapawnGUI()

display(
    game_ui.display()
)